# Failure Analysis: Copilot Opus 4.6

Static analysis of 5 independent runs (101 tasks x 5 runs = 505 result entries).

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))
from utils import get_result_folder, load_results_df

# Load all results from the 5 runs
setup_folder = get_result_folder("bug-fix") / "copilot-opus-4-6"
df = load_results_df(setup_folder)

In [2]:
# Overall success/failure breakdown
total = len(df)
resolved = df["resolved"].sum()
failed = total - resolved
builds_ok = df["build"].sum()
build_failed = total - builds_ok

print(f"Total entries: {total}")
print(f"Resolved: {resolved} ({resolved / total * 100:.1f}%)")
print(f"Failed: {failed} ({failed / total * 100:.1f}%)")
print(f"Build succeeded: {builds_ok} ({builds_ok / total * 100:.1f}%)")
print(f"Build failed: {build_failed} ({build_failed / total * 100:.1f}%)")

Total entries: 505
Resolved: 329 (65.1%)
Failed: 176 (34.9%)
Build succeeded: 491 (97.2%)
Build failed: 14 (2.8%)


## 1. Cross-Run Consistency Analysis

Do the same tasks fail the same way across all 5 runs?

In [3]:
# Pivot: instance_id x run_id -> resolution status
resolution_pivot = df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc="first")

# Calculate consistency metrics
n_runs = resolution_pivot.shape[1]
resolution_pivot["pass_count"] = resolution_pivot.sum(axis=1)
resolution_pivot["fail_count"] = n_runs - resolution_pivot["pass_count"]

# Categorize tasks
always_pass = (resolution_pivot["pass_count"] == n_runs).sum()
always_fail = (resolution_pivot["fail_count"] == n_runs).sum()
flaky = n_runs - always_pass - always_fail

print(f"Tasks that ALWAYS pass (5/5 runs): {always_pass}")
print(f"Tasks that ALWAYS fail (0/5 runs): {always_fail}")
print(f"Flaky tasks (pass sometimes): {len(resolution_pivot) - always_pass - always_fail}")

Tasks that ALWAYS pass (5/5 runs): 51
Tasks that ALWAYS fail (0/5 runs): 22
Flaky tasks (pass sometimes): 28


In [4]:
# Visualize pass/fail consistency
pass_distribution = resolution_pivot["pass_count"].value_counts().sort_index().reset_index()
pass_distribution.columns = ["passes_out_of_5", "task_count"]

fig = px.bar(
    pass_distribution,
    x="passes_out_of_5",
    y="task_count",
    title="Task Resolution Consistency Across 5 Runs",
    labels={"passes_out_of_5": "# of Runs Passed", "task_count": "# of Tasks"},
    text="task_count",
    color="passes_out_of_5",
    color_continuous_scale="RdYlGn",
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

## 2. Localization Analysis: W1 vs Other Folders

The NAV repository contains various localizations. Our evaluation harness only verifies the agent's output in the **W1** folder (by running tests there).

Some fixes might require changes across multiple localizations, but we only evaluate W1. Does the agent touch folders beyond W1, and if so, does it affect resolution rate?

In [5]:
from utils import extract_localizations_from_patch

# Analyze agent outputs for localization touches
df["agent_localizations"] = df["output"].apply(extract_localizations_from_patch)
df["touches_w1"] = df["agent_localizations"].apply(lambda x: "W1" in x)
df["touches_non_w1"] = df["agent_localizations"].apply(lambda x: len(x - {"W1"}) > 0)
df["touches_only_w1"] = df["agent_localizations"].apply(lambda x: x == {"W1"})
df["localization_count"] = df["agent_localizations"].apply(len)

print("=== Localization Folders Touched by Agent ===\n")

# Overall stats
total_with_output = df[df["output"].notna() & (df["output"] != "")].shape[0]
touches_w1_only = df["touches_only_w1"].sum()
touches_non_w1_total = df["touches_non_w1"].sum()

print(f"Total trials with agent output: {total_with_output}")
print(f"Trials touching ONLY W1: {touches_w1_only}")
print(f"Trials touching non-W1 localizations: {touches_non_w1_total}")

# Show distribution of localization counts (exclude 0 - those are timeouts with empty output)
loc_dist = df[df["localization_count"] > 0]["localization_count"].value_counts().sort_index()
print("\nDistribution of # localizations touched:")
for count, n_trials in loc_dist.items():
    pct = 100 * n_trials / len(df)
    print(f"  {count} localizations: {n_trials} trials ({pct:.1f}%)")

=== Localization Folders Touched by Agent ===

Total trials with agent output: 501
Trials touching ONLY W1: 501
Trials touching non-W1 localizations: 0

Distribution of # localizations touched:
  1 localizations: 501 trials (99.2%)


## 3. Interaction Rounds / Excution Time Analysis

How quickly does the agent solve tasks, and do failed tasks drag on?
Calculation of CDF (Cumulative Distribution Function) for success and failure cases.


In [6]:
import plotly.graph_objects as go

df["turn_count_numeric"] = pd.to_numeric(df["turn_count"], errors="coerce").fillna(0).astype(int)

max_turns = min(df["turn_count_numeric"].max(), 250)

success_df = df[df["resolved"]]
failure_df = df[~df["resolved"]]

success_counts = success_df["turn_count_numeric"].value_counts().sort_index()
success_cdf = success_counts.cumsum() / len(success_df) * 100

failure_counts = failure_df["turn_count_numeric"].value_counts().sort_index()
failure_cdf = failure_counts.cumsum() / len(failure_df) * 100

full_range = range(1, max_turns + 1)
success_cdf_filled = success_cdf.reindex(full_range, method="ffill").fillna(0)
failure_cdf_filled = failure_cdf.reindex(full_range, method="ffill").fillna(0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=success_cdf_filled.index, y=success_cdf_filled, mode="lines", name=f"Success (n={len(success_df)})", fill="tozeroy", line={"color": "green"}))
fig.add_trace(go.Scatter(x=failure_cdf_filled.index, y=failure_cdf_filled, mode="lines", name=f"Failure (n={len(failure_df)})", fill="tozeroy"))

# Add median and 80th percentile markers for success
turn_50 = success_cdf_filled[success_cdf_filled >= 50].first_valid_index()
turn_80 = success_cdf_filled[success_cdf_filled >= 80].first_valid_index()
if turn_50:
    fig.add_vline(x=turn_50, line_dash="dash", annotation_text=f"Median: {turn_50}")
if turn_80:
    fig.add_vline(x=turn_80, line_dash="dash", annotation_text=f"80%: {turn_80}")

fig.update_layout(
    title="Cumulative Distribution Function of Interaction Rounds", xaxis_title="Number of Interaction Rounds", yaxis_title="Cumulative Tasks (%)", yaxis={"range": [0, 105]}, hovermode="x unified"
)
fig.show()

In [7]:
# Execution time CDF (same approach as turn count CDF above)
df["execution_time_numeric"] = pd.to_numeric(df["execution_time"], errors="coerce").fillna(0)

max_time = int(df["execution_time_numeric"].max()) + 1

success_time = df[df["resolved"]]["execution_time_numeric"]
failure_time = df[~df["resolved"]]["execution_time_numeric"]

# Build CDFs using 1-second resolution
time_range = range(1, max_time + 1)

success_time_counts = success_time.apply(int).value_counts().sort_index()
success_time_cdf = (success_time_counts.cumsum() / len(success_time) * 100).reindex(time_range, method="ffill").fillna(0)

failure_time_counts = failure_time.apply(int).value_counts().sort_index()
failure_time_cdf = (failure_time_counts.cumsum() / len(failure_time) * 100).reindex(time_range, method="ffill").fillna(0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=success_time_cdf.index, y=success_time_cdf, mode="lines", name=f"Success (n={len(success_time)})", fill="tozeroy", line={"color": "green"}))
fig.add_trace(go.Scatter(x=failure_time_cdf.index, y=failure_time_cdf, mode="lines", name=f"Failure (n={len(failure_time)})", fill="tozeroy"))

# Percentile markers for success
t50 = success_time_cdf[success_time_cdf >= 50].first_valid_index()
t80 = success_time_cdf[success_time_cdf >= 80].first_valid_index()
if t50:
    fig.add_vline(x=t50, line_dash="dash", annotation_text=f"Median: {t50}s")
if t80:
    fig.add_vline(x=t80, line_dash="dash", annotation_text=f"80%: {t80}s")

fig.update_layout(
    title="Cumulative Distribution Function of Execution Time", xaxis_title="Execution Time (seconds)", yaxis_title="Cumulative Tasks (%)", yaxis={"range": [0, 105]}, hovermode="x unified"
)
fig.show()

## 4. Tool Usage Patterns: Success vs Failure

Compare tool usage between resolved and failed tasks. Are failing tasks stuck in exploration loops (excessive grep/view) with too few edits?

In [8]:
import json

from utils import expand_tool_usage

# Some tool_usage values are JSON strings instead of dicts — normalize them
df["tool_usage"] = df["tool_usage"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

# Expand tool_usage dict into separate columns
tool_df = expand_tool_usage(df)
tool_df["resolved"] = df["resolved"].values

# Average tool usage per outcome
tool_means = tool_df.groupby("resolved").mean().T
tool_means.columns = ["Failed", "Resolved"]
tool_means["Diff (Resolved - Failed)"] = tool_means["Resolved"] - tool_means["Failed"]
tool_means = tool_means.sort_values("Diff (Resolved - Failed)", ascending=False)

print("=== Average Tool Usage per Trial ===\n")
print(tool_means.round(1).to_string())

# Visualize as grouped bar chart
fig = go.Figure()
fig.add_trace(go.Bar(x=tool_means.index, y=tool_means["Resolved"], name="Resolved", marker_color="green"))
fig.add_trace(go.Bar(x=tool_means.index, y=tool_means["Failed"], name="Failed", marker_color="red"))
fig.update_layout(title="Average Tool Usage: Resolved vs Failed", xaxis_title="Tool", yaxis_title="Avg Calls per Trial", barmode="group", hovermode="x unified")
fig.show()

=== Average Tool Usage per Trial ===

               Failed  Resolved  Diff (Resolved - Failed)
read_agent        0.0       0.0                      -0.0
update_todo       2.7       2.6                      -0.1
create            0.2       0.2                      -0.1
store_memory      0.1       0.0                      -0.1
report_intent     2.2       2.1                      -0.1
powershell        0.5       0.4                      -0.1
edit              2.2       2.1                      -0.2
glob              0.6       0.4                      -0.2
task              1.8       1.6                      -0.2
view              9.9       7.8                      -2.1
grep              8.4       5.5                      -2.9


## 5. Flaky Task Deep-Dive

Tasks that pass in some runs but fail in others. By comparing successful vs failed runs of the *same task*, we control for task difficulty and isolate what the agent did differently.

In [9]:
# Identify flaky tasks (pass in some runs, fail in others)
task_pass_counts = df.groupby("instance_id")["resolved"].agg(["sum", "count"])
task_pass_counts.columns = ["pass_count", "total_runs"]
flaky_tasks = task_pass_counts[(task_pass_counts["pass_count"] > 0) & (task_pass_counts["pass_count"] < task_pass_counts["total_runs"])]

print(f"Flaky tasks: {len(flaky_tasks)} / {len(task_pass_counts)}")
print("Pass count distribution among flaky tasks:")
print(flaky_tasks["pass_count"].value_counts().sort_index().to_string())

# Filter df to only flaky task runs
flaky_df = df[df["instance_id"].isin(flaky_tasks.index)].copy()
flaky_df["turn_count_numeric"] = pd.to_numeric(flaky_df["turn_count"], errors="coerce").fillna(0).astype(int)
flaky_df["execution_time_numeric"] = pd.to_numeric(flaky_df["execution_time"], errors="coerce").fillna(0)

# Compare metrics between successful and failed runs of flaky tasks
flaky_success = flaky_df[flaky_df["resolved"]]
flaky_failure = flaky_df[~flaky_df["resolved"]]

print("\n=== Flaky Tasks: Successful vs Failed Runs ===\n")
print(f"Successful runs: {len(flaky_success)} | Failed runs: {len(flaky_failure)}")
print(f"\nAvg turn count  — Success: {flaky_success['turn_count_numeric'].mean():.1f} | Failure: {flaky_failure['turn_count_numeric'].mean():.1f}")
print(f"Avg exec time   — Success: {flaky_success['execution_time_numeric'].mean():.0f}s | Failure: {flaky_failure['execution_time_numeric'].mean():.0f}s")

Flaky tasks: 28 / 101
Pass count distribution among flaky tasks:
pass_count
1    7
2    5
3    7
4    9

=== Flaky Tasks: Successful vs Failed Runs ===

Successful runs: 74 | Failed runs: 66

Avg turn count  — Success: 72.4 | Failure: 56.8
Avg exec time   — Success: 420s | Failure: 463s


In [10]:
# Tool usage comparison for flaky tasks: successful runs vs failed runs
flaky_tool_success = expand_tool_usage(flaky_success)
flaky_tool_failure = expand_tool_usage(flaky_failure)

# Align columns
all_tools = sorted(set(flaky_tool_success.columns) | set(flaky_tool_failure.columns))
flaky_tool_success = flaky_tool_success.reindex(columns=all_tools, fill_value=0)
flaky_tool_failure = flaky_tool_failure.reindex(columns=all_tools, fill_value=0)

flaky_tool_comparison = pd.DataFrame(
    {
        "Resolved": flaky_tool_success.mean(),
        "Failed": flaky_tool_failure.mean(),
    }
)
flaky_tool_comparison["Diff"] = flaky_tool_comparison["Resolved"] - flaky_tool_comparison["Failed"]
flaky_tool_comparison = flaky_tool_comparison.sort_values("Diff", ascending=False)

print("=== Flaky Tasks: Avg Tool Usage (Resolved vs Failed Runs) ===\n")
print(flaky_tool_comparison.round(1).to_string())

# Visualize
fig = go.Figure()
fig.add_trace(go.Bar(x=flaky_tool_comparison.index, y=flaky_tool_comparison["Resolved"], name="Resolved Runs", marker_color="green"))
fig.add_trace(go.Bar(x=flaky_tool_comparison.index, y=flaky_tool_comparison["Failed"], name="Failed Runs", marker_color="red"))
fig.update_layout(title="Flaky Tasks: Tool Usage in Resolved vs Failed Runs", xaxis_title="Tool", yaxis_title="Avg Calls per Trial", barmode="group", hovermode="x unified")
fig.show()

=== Flaky Tasks: Avg Tool Usage (Resolved vs Failed Runs) ===

               Resolved  Failed  Diff
view               12.7    10.4   2.3
grep                9.6     7.7   1.9
update_todo         2.8     2.5   0.3
powershell          0.9     0.7   0.3
task                2.0     1.8   0.2
report_intent       2.3     2.2   0.1
create              0.2     0.1   0.1
edit                2.3     2.3  -0.0
store_memory        0.0     0.2  -0.1
glob                0.4     0.6  -0.2


## 6. AL Object Type Analysis

Use the files touched by gold patch, to label each task by AL object type. This answers a question: which kinds of benchmark tasks are easier or harder for the agent to solve?

To avoid overstating noisy categories, we report task-level mean resolution rate by object type and call out tasks that touch multiple object types separately.

In [11]:
import re

from utils import DATASET_PATH

from bcbench.dataset import BugFixEntry

# Extract AL object types from gold patch filenames.
# AL files usually follow <Name>.<ObjectType>.al (for example, SalesPost.Codeunit.al).
AL_FILE_PATTERN = re.compile(r"^diff --git a/.*\.([a-zA-Z]+)\.al b/.*\.([a-zA-Z]+)\.al$", re.MULTILINE)


def extract_al_object_types_from_patch(patch: str) -> list[str]:
    if not isinstance(patch, str) or not patch:
        return []

    object_types = {left.capitalize() for left, right in AL_FILE_PATTERN.findall(patch) if left == right}
    return sorted(object_types)


dataset_entries = BugFixEntry.load(DATASET_PATH)
gold_object_types = {entry.instance_id: extract_al_object_types_from_patch(entry.patch) for entry in dataset_entries}

task_df = (
    df.groupby("instance_id")["resolved"]
    .agg(success_rate="mean", resolved_runs="sum", total_runs="count")
    .reset_index()
    .assign(
        success_rate=lambda frame: frame["success_rate"] * 100,
        al_object_types=lambda frame: frame["instance_id"].map(gold_object_types).apply(lambda value: value or ["Unknown"]),
    )
)
task_df["object_type_count"] = task_df["al_object_types"].apply(len)

single_type_tasks = task_df[task_df["object_type_count"] == 1].explode("al_object_types")
single_type_stats = (
    single_type_tasks.groupby("al_object_types")
    .agg(
        total_tasks=("instance_id", "count"),
        mean_success_rate=("success_rate", "mean"),
    )
    .reset_index()
    .sort_values(["mean_success_rate", "total_tasks"], ascending=[False, False])
)

multi_type_tasks = task_df[task_df["object_type_count"] > 1]

print("=== AL Object Type Performance (Gold Patch, Single-Type Tasks) ===\n")
print(single_type_stats.to_string(index=False, float_format="%.1f"))
print()
print(f"Single-type tasks: {len(single_type_tasks)} / {len(task_df)}")
print(f"Multi-type tasks: {len(multi_type_tasks)} / {len(task_df)}")

plot_stats = single_type_stats[single_type_stats["total_tasks"] >= 5].copy()

if not plot_stats.empty:
    fig = px.bar(
        plot_stats,
        x="al_object_types",
        y="mean_success_rate",
        text="mean_success_rate",
        title="Mean Resolution Rate by Gold AL Object Type (single-type tasks, n>=5)",
        labels={"al_object_types": "AL Object Type", "mean_success_rate": "Mean Resolution Rate (%)"},
        color="mean_success_rate",
        color_continuous_scale="RdYlGn",
        hover_data=["total_tasks"],
    )
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.update_layout(yaxis={"range": [0, 115]}, xaxis_tickangle=-45)
    fig.show()

=== AL Object Type Performance (Gold Patch, Single-Type Tasks) ===

al_object_types  total_tasks  mean_success_rate
         Report            7               71.4
           Page           11               70.9
       Codeunit           46               70.0
          Table           23               64.3
       Tableext            2               10.0

Single-type tasks: 89 / 101
Multi-type tasks: 12 / 101
